## Define helper functions

In [1]:
import os
import pickle
from pathlib import Path

# import required module
import sys

# append the path of the
# parent directory
sys.path.append("..")
 
import env
from helper_functions import check_zeolite_validity, create_structure
from helper_functions import split_samples_to_individual_cdvae, split_samples_to_individual_zdivae
from helper_functions import split_recons_cdvae, split_recons_gt_cdvae
from helper_functions import split_recons_zdivae, split_recons_gt_zdivae
from numpy.linalg import LinAlgError
from pymatgen.analysis.structure_matcher import StructureMatcher

# Load environment variables
env.load_envs()

# Set the cwd to the project root
PROJECT_ROOT: Path = Path(env.get_env("PROJECT_ROOT"))
assert (
    PROJECT_ROOT.exists()
), "You must configure the PROJECT_ROOT environment variable in a .env file!"

os.chdir(PROJECT_ROOT)

cwd = os.getcwd()

## Evaluate reconstructions

### Load evaluations

In [7]:
from utils import retrieve_artifacts_by_name

experiment_name = "sample_recon_zdivae_crf_all_codes_lt50"

artifact_files = retrieve_artifacts_by_name(experiment_name, artifact_type='dataset', project='zeogen', entity='glafk')
reconstructions = None
for file in artifact_files:
    if "recons" in file:
        if "_gt" in file:
            with open(file, "rb") as f:
                ground_truths = pickle.load(f)
        else:
            if reconstructions is None:
                with open(file, "rb") as f:
                    reconstructions = pickle.load(f)



Found run sample_recon_zdivae_crf_all_codes_lt50. Retrieving artifacts...
Run id sbsr7uox


wandb:   1 of 1 files downloaded.  


wandb:   1 of 1 files downloaded.  


wandb:   1 of 1 files downloaded.  


In [6]:
print(reconstructions[0].keys())

dict_keys(['zd', 'zy', 'num_atoms', 'lengths', 'angles', 'frac_coords', 'atom_types', 'domains', 'norm_hoas', 'pred_hoas', 'pred_norm_hoas', 'pred_domains'])


In [8]:
if "zdivae" in experiment_name:
    reconstructions = split_recons_zdivae(reconstructions)
    reconstructions_gt = split_recons_gt_zdivae(ground_truths)
    assert len(reconstructions) == len(reconstructions_gt)
elif "cdvae" in experiment_name:
    reconstructions = split_recons_cdvae(reconstructions)
    reconstructions_gt = split_recons_gt_cdvae(ground_truths)
    assert len(reconstructions) == len(reconstructions_gt)

print(f"Number of reconstructions in this dataset {len(reconstructions)}")

Number of reconstructions in this dataset 1000


In [9]:
total_matches = 0
total_valid = 0
for i, (recon, ground_truth) in enumerate(zip(reconstructions, reconstructions_gt)):
    # Create structures for the reconstruction and ground truth
    coords_recon = recon["frac_coords"]
    lengths_recon = recon["lengths"]
    angles_recon = recon["angles"]

    coords_gt = ground_truth["frac_coords"]
    lengths_gt = ground_truth["lengths"]
    angles_gt = ground_truth["angles"]

    recon = create_structure(coords_recon, lengths_recon, angles_recon)
    ground_truth = create_structure(coords_gt.cpu(), lengths_gt.cpu(), angles_gt.cpu())


    # Initialize StructureMatcher
    matcher = StructureMatcher(ltol=0.3, stol=0.35, angle_tol=5, primitive_cell=True, scale=True, attempt_supercell=False)

    # Compare structures
    is_match = matcher.fit(recon, ground_truth)

    if is_match:
        total_matches += 1
        print(f"The two structures from reconstruction {i} are considered equivalent.")
    else:
        print(f"The two structures from reconstruction {i} are not equivalent.")

    # Check validity
    if check_zeolite_validity(recon):
        print(f"The reconstruction {i} is a valid zeolite.")
        total_valid += 1
    else:
        print(f"The reconstruction {i} is not a valid zeolite.")

    # # Optional: Output the similarity transformation
    # if is_match:
    #     transf = matcher.get_mapping(recon, ground_truth)
    #     print("Transformation matrix:")
    #     print(transf)

print(f"Total matching reconstructions: {total_matches} / {len(reconstructions)}, {(total_matches/len(reconstructions))*100}%")   
print(f"Total valid reconstructions: {total_valid} / {len(reconstructions)}, {(total_valid/len(reconstructions))*100}%")

The two structures from reconstruction 0 are not equivalent.
The reconstruction 0 is not a valid zeolite.
The two structures from reconstruction 1 are not equivalent.
The reconstruction 1 is not a valid zeolite.
The two structures from reconstruction 2 are not equivalent.
The reconstruction 2 is not a valid zeolite.
The two structures from reconstruction 3 are not equivalent.
The reconstruction 3 is not a valid zeolite.
The two structures from reconstruction 4 are not equivalent.
The reconstruction 4 is not a valid zeolite.
The two structures from reconstruction 5 are not equivalent.
The reconstruction 5 is not a valid zeolite.
The two structures from reconstruction 6 are not equivalent.
The reconstruction 6 is not a valid zeolite.
The two structures from reconstruction 7 are not equivalent.
The reconstruction 7 is not a valid zeolite.
The two structures from reconstruction 8 are not equivalent.
The reconstruction 8 is not a valid zeolite.
The two structures from reconstruction 9 are n

In [11]:
# Evaluate type accuracies

from statistics import mean
import numpy as np

type_accuracies = []
al_accuracies = []
for i, (recon, ground_truth) in enumerate(zip(reconstructions, reconstructions_gt)):
    try:
        overall_accuracy = (np.array(recon["atom_types"]) == ground_truth["atom_types"].cpu().numpy())
    except ValueError:
        print(f"Reconstruction {i} hsa different number of atoms in the ground truth. Skipping...")
        continue

    type_accuracy = overall_accuracy.mean()
    type_accuracies.extend([type_accuracy])

    al_mask = ground_truth["atom_types"].cpu().numpy() == 13
    al_accuracy = overall_accuracy[al_mask].mean()
    al_accuracies.extend([al_accuracy])

print(mean(type_accuracies))
print(mean(al_accuracies))
    

Reconstruction 446 hsa different number of atoms in the ground truth. Skipping...
Reconstruction 583 hsa different number of atoms in the ground truth. Skipping...
0.8325773286672546
0.0225827918463189


In [12]:
from sklearn.metrics import silhouette_score

if "zdivae" in experiment_name:
    embeddings = [recon["zd"] for recon in reconstructions]
elif "cdvae" in experiment_name:
    embeddings = [recon["z"] for recon in reconstructions]
labels = [recon["domains"] for recon in reconstructions]

# Compute the silhouette score
silhouette_avg = silhouette_score(embeddings, labels)
print(f"Silhouette score: {silhouette_avg}")

Silhouette score: 0.4960331320762634


## Evaluate samples

### Load samples


In [5]:
from utils import retrieve_artifacts_by_name

experiment_name = "sample_recon_base_cdvae_small_lt50"

artifact_files = retrieve_artifacts_by_name(experiment_name, artifact_type='dataset', project='zeogen', entity='glafk')

for file in artifact_files:
    if "samples" in file:
        with open(file, "rb") as f:
            samples = pickle.load(f)


# samples_file = f"samples-{experiment_name}.pickle"

# with open(f"./source/evaluation/experiment_results/samples/{samples_file}", "rb") as f:
#     samples = pickle.load(f)


Found run sample_recon_base_cdvae_small_lt50. Retrieving artifacts...


wandb:   1 of 1 files downloaded.  


wandb:   1 of 1 files downloaded.  


wandb:   1 of 1 files downloaded.  


In [6]:
if "zdivae" in experiment_name:
    samples = split_samples_to_individual_zdivae(samples)
elif "cdvae" in experiment_name:
    samples = split_samples_to_individual_cdvae(samples)

print(f"Number of samples in this dataset {len(samples)}")


Number of samples in this dataset 95


### Evaluate metrics

In [7]:
# Evaluate sample validity
total_valid = 0
for i, sample in enumerate(samples):
    coords = sample["frac_coords"]
    lengths = sample["lengths"]
    angles = sample["angles"]

    structure = create_structure(coords, lengths, angles)

    try:
        zeolite_valid = check_zeolite_validity(structure, tolerance=3.5)
    except LinAlgError as e:
        print(f"Skipping sample {i} due to error in distance matrix. ")
    if zeolite_valid:
        total_valid += 1
        print(f"The sample {i} is a valid zeolite.")
    else:
        print(f"The sample {i} is not a valid zeolite.")

print(f"Total valid samples: {total_valid} / {len(samples)}, {(total_valid/len(samples))*100}%")

The sample 0 is not a valid zeolite.
The sample 1 is not a valid zeolite.
The sample 2 is not a valid zeolite.
The sample 3 is not a valid zeolite.
The sample 4 is not a valid zeolite.
The sample 5 is not a valid zeolite.
The sample 6 is not a valid zeolite.
The sample 7 is not a valid zeolite.
The sample 8 is not a valid zeolite.
The sample 9 is a valid zeolite.
The sample 10 is not a valid zeolite.
The sample 11 is a valid zeolite.
The sample 12 is not a valid zeolite.
The sample 13 is not a valid zeolite.
The sample 14 is not a valid zeolite.
The sample 15 is a valid zeolite.
The sample 16 is a valid zeolite.
The sample 17 is not a valid zeolite.
The sample 18 is a valid zeolite.
The sample 19 is not a valid zeolite.
The sample 20 is not a valid zeolite.
The sample 21 is not a valid zeolite.
The sample 22 is not a valid zeolite.
The sample 23 is a valid zeolite.
The sample 24 is not a valid zeolite.
The sample 25 is not a valid zeolite.
The sample 26 is not a valid zeolite.
The samp

In [37]:
# Calculate predicted domain accuracy for samples

total_correctly_predicted_domains = 0
for sample in samples:
    if sample["domains"] == sample["pred_domains"]:
        total_correctly_predicted_domains += 1 

print(f"Total correctly predicted domains: {total_correctly_predicted_domains} / {len(samples)}, {(total_correctly_predicted_domains/len(samples))*100}%")

Total correctly predicted domains: 480 / 960, 50.0%


In [17]:
from sklearn.metrics import silhouette_score

if "zdivae" in experiment_name:
    embeddings = [sample["zd"] for sample in samples]
elif "cdvae" in experiment_name:
    embeddings = [sample["z"] for sample in samples]
labels = [sample["domains"] for sample in samples]

# Compute the silhouette score
silhouette_avg = silhouette_score(embeddings, labels)
print(f"Silhouette score: {silhouette_avg}")

Silhouette score: -0.022666145116090775
